## Query SQL using Python

Todo: Download files to NERSC using the SciServer Python module

In [1]:
import SciServer
from SciServer import CasJobs, SkyQuery, Files
print('Imported SciServer modules')

import pandas                                # data analysis tools
import numpy as np                           # numerical tools
from datetime import datetime, timedelta     # date and timestamp tools
from pprint import pprint                    # print human-readable output
print('Imported other needed modules')

Imported SciServer modules
Imported other needed modules


In [2]:
import SciServer
from SciServer import Authentication
print('Imported SciServer modules')

# These are the credentials you will use later to manually log in to SciServer.
### The default values are for a test account we created specifically for this notebook.
#### The example will be more useful if you replace with your own username/password, but if you do, 
####   please be careful to keep this notebook private.
Authentication_loginName = 'zzbenjamin94'
Authentication_loginPassword = 'ZZhangSDSS94!'
this_context = "MyDB"    # Your MyDB
print('Set login name and password')

Imported SciServer modules
Set login name and password


In [3]:
manualtoken = ""
manualtoken = Authentication.login(Authentication_loginName, Authentication_loginPassword)
manualtokenvalue = Authentication.token.value

if (manualtoken):
    print("Manual login (providing login and password in notebook) successful!")
    #print("Login token (via manual login): {0:}".format(autotoken))
else:
    print("ERROR: Manual login failed. Please check your commands and try again.")
    print("For help, type:")
    print("help(Authentication)")

Manual login (providing login and password in notebook) successful!


In [4]:
# PYTHON CONVENIENCE FUNCTIONS USEFUL FOR WORKING WITH CASJOBS

def tables_formatted(tables):   # better formatted printing of a tables dictionary (output of get_tables)
# Returns the following information about the tables in your MyDB (as a Python dictionary object):
### Size: size of the table (in kB)
### Name: the name of the table
### Rows: the number of rows the table contains
### Date: the date of the table's creation, as the number of 10-microsecond intervals elapsed 1 AD

    import pandas
    from datetime import datetime
    
    tables = sorted(tables, key=lambda k: k['Name']) # alphabetize by table name
    
    for thistable in tables:
        print('Table name:\t',thistable['Name'])
        print('Rows:\t\t {:,.0f}'.format(thistable['Rows']))
        print('Size (kB):\t {:,.0f} '.format(thistable['Size']))

        cjCreateDate = thistable['Date']
        createsec = cjCreateDate / 10000000  # Divide by 10 million to get seconds elapsed since 1 AD
        firstday = datetime(1, 1, 1, 0, 0)   # Save 1 AD as "firstday"
        created = firstday + timedelta(seconds=createsec)  # Get calendar date on which table was created     
        print('Created time:\t',created.strftime('%Y-%m-%d %H:%M:%S'))
        print('\n')
        

def jobDescriber(jobDescription):
    # Prints the results of the CasJobs job status functions in a human-readable manner
    # Input: the python dictionary returned by getJobStatus(jobId) or waitForJob(jobId)
    # Output: prints the dictionary to screen with readable formatting
    import pandas
    
    if (jobDescription["Status"] == 0):
        status_word = 'Ready'
    elif (jobDescription["Status"] == 1):
        status_word = 'Started'
    elif (jobDescription["Status"] == 2):
        status_word = 'Cancelling'
    elif (jobDescription["Status"] == 3):
        status_word = 'Cancelled'
    elif (jobDescription["Status"] == 4):
        status_word = 'Failed'
    elif (jobDescription["Status"] == 5):
        status_word = 'Finished'
    else:
        status_word = 'Status not found!!!!!!!!!'

    print('JobID: ', jobDescription['JobID'])
    print('Status: ', status_word, ' (', jobDescription["Status"],')')
    print('Target (context being searched): ', jobDescription['Target'])
    print('Message: ', jobDescription['Message'])
    print('Created_Table: ', jobDescription['Created_Table'])
    print('Rows: ', jobDescription['Rows'])
    wait = pandas.to_datetime(jobDescription['TimeStart']) - pandas.to_datetime(jobDescription['TimeSubmit'])
    duration = pandas.to_datetime(jobDescription['TimeEnd']) - pandas.to_datetime(jobDescription['TimeStart'])
    print('Wait time: ',wait.seconds,' seconds')
    print('Query duration: ',duration.seconds, 'seconds')
        
print('Created functions')

Created functions


## Quering new data. This is done. 

In [5]:
# note the space at the end of this string - important
myquery = """
SELECT TOP 10 objid, ra, dec, r
FROM galaxy
WHERE clean = 1
"""

# A badly-formed query
#badquery = "ceci n''est pas une query"   # note substitution of '' for single quote mark

# A valid query that returns no results
#zeroquery = "select top 10 * from specobj where 0=1"

# Execute a query to the DR14 context, return a pandas dataframe, then show it
df = CasJobs.executeQuery(sql=myquery, context="DR14")

In [10]:
myquery = """
SELECT TOP 100
        p.objid,
        p.ra, p.dec, 
        p.dered_g, p.dered_r, modelMagErr_g, modelMagErr_r,
        z.Z,
        z.zErr
    FROM PhotoPrimary AS p
    JOIN Photoz AS z
        ON p.objid = z.objid
    WHERE p.type = 3 AND p.clean = 1
    AND p.modelMag_i < 21
    AND p.ra BETWEEN 0 AND 10
    AND p.dec BETWEEN -11.201826615164308 AND 68.72255623374552 
"""
df = CasJobs.executeQuery(sql=myquery, context="DR8")

In [12]:
df[80:100]

,objid,ra,dec,dered_g,dered_r,modelMagErr_g,modelMagErr_r,Z,zErr
80,1237652900748132790,0.006213,-9.959662,22.13334,20.97120,0.174601,0.091356,0.475807,0.106724
81,1237652900748133136,0.001235,-9.812173,22.37557,20.93112,0.211932,0.086854,0.405878,0.094965
82,1237652900748133141,0.004541,-9.775038,22.67546,21.51759,0.262540,0.138735,0.531675,0.105825
83,1237652901285003562,0.003667,-9.464141,20.66500,19.45385,0.045435,0.022496,0.172238,0.011063
84,1237652901285003567,0.006155,-9.524558,22.24788,20.32377,0.166112,0.042922,0.372706,0.020489
85,1237652942101545608,0.001090,13.920549,23.37012,20.81357,0.548973,0.078422,0.499331,0.080555
86,1237652942101545610,0.001407,13.934664,20.47088,19.94635,0.040282,0.034150,0.084309,0.042785
87,1237652942101545612,0.003449,13.903290,21.42265,19.93170,0.095259,0.034421,0.313488,0.050048
88,1237652942101545621,0.004411,14.085933,22.13174,20.74333,0.136718,0.051940,0.423238,0.053063
89,1237652943712158059,0.005622,15.361602,20.65372,19.88506,0.039136,0.029818,0.311841,0.085163


In [23]:
ra = [i for i in range(0,361,30)]
for ind in range(len(ra)-1):
    tablename = "sdss_overlap_ra" + str(ra[ind]) + '_' + str(ra[ind+1])+'_photoZ'

    # Example of a longer query: get magnitudes and sizes (Petrosian radii) of one million galaxies
    verylongquery = """
    SELECT
      p.objid, p.ra, p.dec,
      p.dered_g, p.dered_r,
      z.Z,
      z.zErr
    INTO mydb.{tablename}
    FROM PhotoPrimary AS p
    JOIN Photoz AS z
        ON p.objid = z.objid
    WHERE p.type = 3 AND p.clean = 1
        AND p.modelMag_i < 21
        AND p.ra BETWEEN {ra_low} AND {ra_high}
        AND p.dec BETWEEN -11.201826615164308 AND 68.72255623374552
    """.format(tablename=tablename, ra_low=ra[ind], ra_high = ra[ind+1])
    
    print('Submitting query:\n',verylongquery)
    print('\n')
    
    thisjobid = CasJobs.submitJob(sql=verylongquery, context="DR8")
    
    print('Job submitted with jobId = ',thisjobid)
    print('\n')
    
    #waited = CasJobs.waitForJob(jobId=thisjobid)      # waited is a dummy variable; just print wait msg
    #jobDescription = CasJobs.getJobStatus(thisjobid)
    
    print('\n')
    print('Information about the job:')
    
    #pprint(jobDescription)
    #jobDescriber(jobDescription)

Submitting query:
 
    SELECT
      p.objid, p.ra, p.dec,
      p.dered_g, p.dered_r,
      z.Z,
      z.zErr
    INTO mydb.sdss_overlap_ra0_30_photoZ
    FROM PhotoPrimary AS p
    JOIN Photoz AS z
        ON p.objid = z.objid
    WHERE p.type = 3 AND p.clean = 1
        AND p.modelMag_i < 21
        AND p.ra BETWEEN 0 AND 30
        AND p.dec BETWEEN -11.201826615164308 AND 68.72255623374552
    


Job submitted with jobId =  79385513




Information about the job:
Submitting query:
 
    SELECT
      p.objid, p.ra, p.dec,
      p.dered_g, p.dered_r,
      z.Z,
      z.zErr
    INTO mydb.sdss_overlap_ra30_60_photoZ
    FROM PhotoPrimary AS p
    JOIN Photoz AS z
        ON p.objid = z.objid
    WHERE p.type = 3 AND p.clean = 1
        AND p.modelMag_i < 21
        AND p.ra BETWEEN 30 AND 60
        AND p.dec BETWEEN -11.201826615164308 AND 68.72255623374552
    


Job submitted with jobId =  79385514




Information about the job:
Submitting query:
 
    SELECT
      p.objid, p.ra, 

## Download Files onto SCRATCH Space

In [ ]:
##
download_path = 'pscratch/sd/z/zzhang13/SDSS/sdss_dr8_galaxies/'

In [ ]:
from SciServer import Authentication, Jobs, CasJobs
import requests
import os

# --- Login ---
#Authentication.Login('zzbenjamin94', 'ZZhangSDSS94!')

# --- Request a CSV export of MyDB.my_export ---
fileInfo = CasJobs.getOutputFile(
    "sdss_overlap_ra100_110",                 # table name
    "fits"                        # format: csv, fits, tsb
)

# fileInfo gives you a temporary HTTPS URL from casjobs
url = fileInfo["url"]
local_filename = "sdss_overlap_ra100_110.fits"

# --- Download directly ---
r = requests.get(url, stream=True)
with open(download_path + local_filename, 'wb') as f:
    for chunk in r.iter_content(chunk_size=8192):
        if chunk:
            f.write(chunk)

print("Downloaded:", local_filename)